In [1]:
import pandas as pd
import numpy as np

# Load original datasets
ball = pd.read_csv("../data/ball_by_ball.csv")
matches = pd.read_csv("../data/matches.csv")
competitions = pd.read_csv("../data/competitions.csv")

# Merge match and competition information
ball_matches = ball.merge(
    matches,
    on="match_id",
    how="left"
)

ball_matches = ball_matches.merge(
    competitions,
    on="comp_id",
    how="left"
)

print(ball_matches.shape)

(200999, 57)


In [2]:
# Identifying IPL and T20I competitions
valid_competitions = competitions[
    (competitions["comp_name"].str.contains(
        "IPL", case=False, na=False
    )) |
    (competitions["match_format"] == "T20I")
]

valid_comp_ids = valid_competitions["comp_id"].unique()

# Filter ball-by-ball data
df = ball_matches[
    ball_matches["comp_id"].isin(valid_comp_ids)
].copy()

print("Filtered dataset:", df.shape)

print(df["match_format"].value_counts())
print(df["comp_name"].nunique(), "competitions")

Filtered dataset: (194014, 57)
match_format
T20I    106327
T20      87687
Name: count, dtype: int64
136 competitions


In [3]:
print("Before filtering:", df.shape)

df = df[df["innings"].isin([1, 2])].copy()

print("After filtering:", df.shape)

print(df["innings"].value_counts())

Before filtering: (194014, 57)
After filtering: (193988, 57)
innings
1    102054
2     91934
Name: count, dtype: int64


In [4]:
# Creating separate columns for different types of runs
# Fill missing values in run columns
run_columns = [
    "runs",
    "extras1_runs",
    "extras_2_runs"
]

df[run_columns] = df[run_columns].fillna(0)

# Batter runs
df["batter_runs"] = df["runs"]

# Extras recorded
df["extras_runs"] = (
    df["extras1_runs"] +
    df["extras_2_runs"]
)

# Total team runs
df["total_runs"] = (
    df["batter_runs"] +
    df["extras_runs"]
)

display(
    df[
        [
            "runs",
            "extras_type",
            "extras1_runs",
            "extras_2_runs",
            "batter_runs",
            "extras_runs",
            "total_runs"
        ]
    ].head(15)
)

,runs,extras_type,extras1_runs,extras_2_runs,batter_runs,extras_runs,total_runs
0,0,NaN,0,0,0,0,0
1,0,NaN,0,0,0,0,0
2,0,NaN,0,0,0,0,0
3,0,NaN,0,0,0,0,0
4,6,NaN,0,0,6,0,6
5,0,NaN,0,0,0,0,0
6,0,NaN,0,0,0,0,0
7,0,NaN,0,0,0,0,0
8,0,NaN,0,0,0,0,0
9,4,NaN,0,0,4,0,4


In [5]:
# Identifying legal, illegal and special-event records

df["ball_status"] = "legal"

# Wides and Overstep (no-balls)
df.loc[
    df["extras_type"].isin(["wide", "Overstep"]),
    "ball_status"
] = "illegal"

# Special records requiring separate treatment
df.loc[
    df["extras_type"].isin(["other", "Penalty"]),
    "ball_status"
] = "review"

print(df["ball_status"].value_counts(dropna=False))


ball_status
legal      185691
illegal      8037
review        260
Name: count, dtype: int64


In [6]:
import os

os.makedirs("../outputs", exist_ok=True)

df.to_csv(
    "../outputs/filtered_deliveries.csv",
    index=False
)

print("Filtered dataset saved successfully!")

Filtered dataset saved successfully!


In [7]:
print(
    df[df["match_format"] == "T20"]["comp_name"]
    .value_counts()
)

comp_name
IPL 2022    17909
IPL 2023    17863
IPL 2026    17523
IPL 2025    17275
IPL 2024    17103
Name: count, dtype: int64


In [8]:
display(
    df.groupby(
        "extras_type",
        dropna=False
    ).agg(
        deliveries=("match_id", "count"),
        batter_runs=("batter_runs", "sum"),
        extras=("extras_runs", "sum"),
        total_runs=("total_runs", "sum")
    ).sort_values("deliveries", ascending=False)
)

,deliveries,batter_runs,extras,total_runs
extras_type,,,,
NaN,182525,259841,2,259843
wide,7472,0,8880,8880
legbyes,2661,0,3500,3500
Overstep,565,834,613,1447
byes,505,0,960,960
other,259,407,293,700
Penalty,1,0,5,5


In [9]:
print(matches.columns.tolist())

['match_id', 'comp_id', 'match_number', 'match_date', 'stage', 'venue_id', 'venue_country', 'home_team_id', 'home_team', 'away_team_id', 'away_team', 'home_team_captain_id', 'away_team_captain_id', 'toss_win_team_id', 'toss_opted', 'man_of_the_match', 'first_inn_revised_overs', 'second_inn_revised_overs', 'revised_target', 'knock_out', 'win_team_id', 'result']


In [10]:
df.to_csv(
    "../outputs/ipl_t20i_regular_deliveries.csv",
    index=False
)

In [11]:
# Examine wide deliveries

wide_analysis = (
    df[df["extras_type"] == "wide"]
    .groupby(
        ["extras1_runs", "extras_2_runs"],
        dropna=False
    )
    .agg(
        deliveries=("match_id", "count"),
        batter_runs=("batter_runs", "sum"),
        total_runs=("total_runs", "sum")
    )
    .reset_index()
    .sort_values("deliveries", ascending=False)
)

display(wide_analysis)

,extras1_runs,extras_2_runs,deliveries,batter_runs,total_runs
0,1,0,6887,0,6887
1,2,0,286,0,572
4,5,0,261,0,1305
2,3,0,36,0,108
3,4,0,2,0,8


In [12]:
# Inspect wide deliveries with more than one extra run

wide_multiple = df[
    (df["extras_type"] == "wide") &
    (df["extras_runs"] > 1)
]

display(
    wide_multiple[
        [
            "match_id",
            "innings",
            "striker_name",
            "bowler_name",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "total_runs"
        ]
    ].head(20)
)

,match_id,innings,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,total_runs
352,13512,2,Brandon King,Reece Topley,0,2,0,2
559,13513,1,Rovman Powell,Liam Livingstone,0,5,0,5
607,13513,2,Jason Roy,Jason Holder,0,5,0,5
721,13513,2,Tymal Mills,Romario Shepherd,0,2,0,2
1104,13515,1,Rovman Powell,Chris Jordan,0,2,0,2
1589,13523,1,Venkatesh Iyer,Jason Holder,0,2,0,2
1615,13523,2,Kyle Mayers,Bhuvneshwar Kumar,0,5,0,5
1689,13523,2,Rovman Powell,Ravi Bishnoi,0,2,0,2
2884,13615,2,Glenn Maxwell,Jeffrey Vandersay,0,2,0,2
3057,13616,1,Daniel Sams,Chamika Karunaratne,0,2,0,2


In [13]:
display(
    df[df["extras_type"] == "other"][
        [
            "match_id",
            "innings",
            "striker_name",
            "bowler_name",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "total_runs"
        ]
    ].head(20)
)

,match_id,innings,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,total_runs
532,13513,1,Rovman Powell,Tymal Mills,0,1,0,1
1079,13515,1,Rovman Powell,Saqib Mahmood,0,1,0,1
1226,13515,2,Sam Billings,Sheldon Cottrell,2,1,0,3
2588,13614,1,Dasun Shanaka,Mitchell Starc,0,1,4,5
2609,13614,2,Ashton Agar,Nuwan Thushara,0,1,0,1
3309,14094,1,Mahendra Singh Dhoni,Shivam Mavi,6,1,0,7
4413,14098,2,Bhuvneshwar Kumar,Nathan Coulter Nile,0,1,0,1
5012,14101,1,Kagiso Rabada,Tim Southee,0,1,0,1
5365,14102,2,Kieron Pollard,Prasidh Krishna,0,1,0,1
6575,14107,2,Patrick Cummins,Daniel Sams,2,1,0,3


In [14]:
display(
    df[df["extras_type"] == "Penalty"][
        [
            "match_id",
            "innings",
            "striker_name",
            "bowler_name",
            "runs",
            "extras1_runs",
            "extras_2_runs"
        ]
    ]
)

,match_id,innings,striker_name,bowler_name,runs,extras1_runs,extras_2_runs
34203,16753,2,Harry Tector,Fareed Malik,0,5,0


In [15]:
other_analysis = (
    df[df["extras_type"] == "other"]
    .groupby(
        ["extras1_runs", "extras_2_runs", "is_wicket"],
        dropna=False
    )
    .agg(
        deliveries=("match_id", "count"),
        batter_runs=("batter_runs", "sum"),
        total_runs=("total_runs", "sum")
    )
    .reset_index()
)

display(other_analysis)

,extras1_runs,extras_2_runs,is_wicket,deliveries,batter_runs,total_runs
0,1,0,False,244,407,651
1,1,0,True,2,0,2
2,1,1,False,6,0,12
3,1,4,False,7,0,35


In [16]:
other_deliveries = df[
    df["extras_type"] == "other"
]

display(
    other_deliveries[
        [
            "match_id",
            "innings",
            "overid",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket"
        ]
    ].head(20)
)

,match_id,innings,overid,innings_ball_num,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,is_wicket
532,13513,1,10,57,Rovman Powell,Tymal Mills,0,1,0,False
1079,13515,1,16,99,Rovman Powell,Saqib Mahmood,0,1,0,False
1226,13515,2,19,119,Sam Billings,Sheldon Cottrell,2,1,0,False
2588,13614,1,18,111,Dasun Shanaka,Mitchell Starc,0,1,4,False
2609,13614,2,2,7,Ashton Agar,Nuwan Thushara,0,1,0,False
3309,14094,1,19,119,Mahendra Singh Dhoni,Shivam Mavi,6,1,0,False
4413,14098,2,20,119,Bhuvneshwar Kumar,Nathan Coulter Nile,0,1,0,False
5012,14101,1,17,105,Kagiso Rabada,Tim Southee,0,1,0,False
5365,14102,2,19,122,Kieron Pollard,Prasidh Krishna,0,1,0,True
6575,14107,2,16,95,Patrick Cummins,Daniel Sams,2,1,0,False


In [17]:
display(
    df[df["extras_type"] == "Penalty"].T
)

,34203
match_id,16753
innings,2
overid,6
over_ball_num,4
innings_ball_num,34
...,...
season,2022
batter_runs,0
extras_runs,5
total_runs,5


In [18]:
display(
    df.groupby(
        "extras_type",
        dropna=False
    ).agg(
        deliveries=("match_id", "count"),
        batter_runs=("batter_runs", "sum"),
        extras_runs=("extras_runs", "sum"),
        wickets=("is_wicket", "sum")
    ).sort_values("deliveries", ascending=False)
)

,deliveries,batter_runs,extras_runs,wickets
extras_type,,,,
NaN,182525,259841,2,10431
wide,7472,0,8880,39
legbyes,2661,0,3500,0
Overstep,565,834,613,5
byes,505,0,960,1
other,259,407,293,2
Penalty,1,0,5,0


In [19]:
other_records = df[
    df["extras_type"] == "other"
]

display(
    other_records[
        [
            "match_id",
            "innings",
            "overid",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type"
        ]
    ].head(30)
)

,match_id,innings,overid,innings_ball_num,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,is_wicket,dismissal_type
532,13513,1,10,57,Rovman Powell,Tymal Mills,0,1,0,False,NaN
1079,13515,1,16,99,Rovman Powell,Saqib Mahmood,0,1,0,False,NaN
1226,13515,2,19,119,Sam Billings,Sheldon Cottrell,2,1,0,False,NaN
2588,13614,1,18,111,Dasun Shanaka,Mitchell Starc,0,1,4,False,NaN
2609,13614,2,2,7,Ashton Agar,Nuwan Thushara,0,1,0,False,NaN
3309,14094,1,19,119,Mahendra Singh Dhoni,Shivam Mavi,6,1,0,False,NaN
4413,14098,2,20,119,Bhuvneshwar Kumar,Nathan Coulter Nile,0,1,0,False,NaN
5012,14101,1,17,105,Kagiso Rabada,Tim Southee,0,1,0,False,NaN
5365,14102,2,19,122,Kieron Pollard,Prasidh Krishna,0,1,0,True,RunOut
6575,14107,2,16,95,Patrick Cummins,Daniel Sams,2,1,0,False,NaN


In [20]:
print([
    col for col in df.columns
    if any(word in col.lower()
           for word in [
               "legal", "wide", "noball",
               "no_ball", "delivery"
           ])
])

['delivery_style', 'delivery_ist_time']


In [21]:
# Final legal-ball classification

df["is_legal_ball"] = ~df["extras_type"].isin(
    ["wide", "Overstep", "Penalty"]
)

df["ball_status"] = np.where(
    df["is_legal_ball"],
    "legal",
    "illegal"
)

print(df["ball_status"].value_counts())


ball_status
legal      185950
illegal      8038
Name: count, dtype: int64


In [22]:
# Inspect the penalty record in detail

penalty_record = df[
    df["extras_type"] == "Penalty"
]

display(
    penalty_record[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras1_runs",
            "extras_2_runs"
        ]
    ].T
)

,34203
match_id,16753
innings,2
overid,6
over_ball_num,4
innings_ball_num,34
striker_name,Harry Tector
bowler_name,Fareed Malik
runs,0
extras1_runs,5
extras_2_runs,0


In [23]:
display(
    df[
        (df["extras_type"] == "other") &
        (df["is_wicket"] == True)
    ][
        [
            "match_id",
            "innings",
            "striker_name",
            "bowler_name",
            "dismissal_type",
            "runs",
            "extras1_runs",
            "extras_2_runs"
        ]
    ]
)

,match_id,innings,striker_name,bowler_name,dismissal_type,runs,extras1_runs,extras_2_runs
5365,14102,2,Kieron Pollard,Prasidh Krishna,RunOut,0,1,0
15456,14145,1,Namboori Tilak Varma,Lockie Ferguson,RunOut,0,1,0


In [24]:
# Inspect the complete innings containing the penalty record

penalty_context = df[
    (df["match_id"] == 16753) &
    (df["innings"] == 2)
].sort_values("innings_ball_num")

display(
    penalty_context[
        [
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras_type",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket"
        ]
    ].iloc[25:42]
)

,overid,over_ball_num,innings_ball_num,striker_name,bowler_name,runs,extras_type,extras1_runs,extras_2_runs,is_wicket
34195,5,2,26,Harry Tector,Rashid Khan,1,NaN,0,0,False
34196,5,3,27,George Dockrell,Rashid Khan,1,NaN,0,0,False
34197,5,4,28,Harry Tector,Rashid Khan,1,NaN,0,0,False
34198,5,5,29,George Dockrell,Rashid Khan,1,NaN,0,0,False
34199,5,6,30,Harry Tector,Rashid Khan,1,NaN,0,0,False
34200,6,1,31,Harry Tector,Fareed Malik,1,NaN,0,0,False
34201,6,2,32,George Dockrell,Fareed Malik,2,NaN,0,0,False
34202,6,3,33,George Dockrell,Fareed Malik,1,NaN,0,0,False
34203,6,4,34,Harry Tector,Fareed Malik,0,Penalty,5,0,False
34204,6,5,35,Harry Tector,Fareed Malik,4,NaN,0,0,False


In [25]:
other_wicket_context = df[
    (df["match_id"].isin([14102, 14145])) &
    (df["extras_type"] == "other")
]

display(other_wicket_context)

,match_id,innings,overid,over_ball_num,innings_ball_num,batting_team_id,bowling_team_id,striker_id,striker_name,non_striker_id,...,win_team_id,result,comp_name,match_format,season,batter_runs,extras_runs,total_runs,ball_status,is_legal_ball
5365,14102,2,19,9,122,2,3,494,Kieron Pollard,5034,...,3.0,Rajasthan Royals Won by 23 Runs,IPL 2022,T20,2022,0,1,1,legal,True
15456,14145,1,19,3,114,2,3508,7897,Namboori Tilak Varma,6396,...,2.0,Mumbai Indians won by 5 runs,IPL 2022,T20,2022,0,1,1,legal,True


In [26]:
# Inspect the competitions currently included

print(
    df[["comp_name", "match_format"]]
    .drop_duplicates()
    .sort_values(["match_format", "comp_name"])
    .to_string(index=False)
)

                                              comp_name match_format
                                               IPL 2022          T20
                                               IPL 2023          T20
                                               IPL 2024          T20
                                               IPL 2025          T20
                                               IPL 2026          T20
             Afghanistan in Bangladesh T20I Series 2023         T20I
                  Afghanistan in India T20I Series 2024         T20I
                Afghanistan in Ireland T20I Series 2022         T20I
              Afghanistan in Sri Lanka T20I Series 2024         T20I
               Afghanistan in Zimbabwe T20I Series 2022         T20I
               Afghanistan in Zimbabwe T20I Series 2024         T20I
               Afghanistan in Zimbabwe T20I Series 2025         T20I
              Afghanistan v Bangladesh T20I Series 2025         T20I
                 Afghanistan v Ire

In [27]:
# Check delivery styles for unusual extras

display(
    df[df["extras_type"].isin(["other", "Penalty"])]
    .groupby(
        ["extras_type", "delivery_style"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
)

,extras_type,delivery_style,count
0,Penalty,Full Toss,1
1,other,Angled In,2
2,other,Arm ball,1
3,other,Back of Hand,4
4,other,Beamer,115
5,other,Bouncer,16
6,other,Carrom,1
7,other,Flipper,1
8,other,Full Toss,70
9,other,Googly,2


In [28]:

# Sort deliveries chronologically within each innings

df = df.sort_values(
    ["match_id", "innings", "innings_ball_num"]
).copy()

# Cumulative team runs
df["cumulative_runs"] = (
    df.groupby(["match_id", "innings"])["total_runs"]
    .cumsum()
)

# Total wickets lost
df["cumulative_wickets"] = (
    df.groupby(["match_id", "innings"])["is_wicket"]
    .cumsum()
)

# Cumulative legal deliveries
df["legal_balls_bowled"] = (
    df["ball_status"].eq("legal")
    .groupby([df["match_id"], df["innings"]])
    .cumsum()
)

# Wickets remaining
df["wickets_remaining"] = (
    10 - df["cumulative_wickets"]
)

display(
    df[
        [
            "match_id",
            "innings",
            "striker_name",
            "total_runs",
            "cumulative_runs",
            "cumulative_wickets",
            "legal_balls_bowled",
            "wickets_remaining"
        ]
    ].head(20)
)


,match_id,innings,striker_name,total_runs,cumulative_runs,cumulative_wickets,legal_balls_bowled,wickets_remaining
0,13511,1,Jason Roy,0,0,0,1,10
1,13511,1,Jason Roy,0,0,0,2,10
2,13511,1,Jason Roy,0,0,0,3,10
3,13511,1,Jason Roy,0,0,0,4,10
4,13511,1,Jason Roy,6,6,0,5,10
5,13511,1,Jason Roy,0,6,1,6,9
6,13511,1,Tom Banton,0,6,1,7,9
7,13511,1,Tom Banton,0,6,1,8,9
8,13511,1,Tom Banton,0,6,1,9,9
9,13511,1,Tom Banton,4,10,1,10,9


In [29]:
df["current_run_rate"] = np.where(
    df["legal_balls_bowled"] > 0,
    df["cumulative_runs"] * 6 /
    df["legal_balls_bowled"],
    np.nan
)

In [30]:
display(
    df[
        [
            "match_id",
            "innings",
            "cumulative_runs",
            "cumulative_wickets",
            "legal_balls_bowled",
            "wickets_remaining",
            "current_run_rate"
        ]
    ].tail(20)
)

,match_id,innings,cumulative_runs,cumulative_wickets,legal_balls_bowled,wickets_remaining,current_run_rate
200979,55561,2,46,3,30,7,9.200000
200980,55561,2,48,3,31,7,9.290323
200981,55561,2,48,3,32,7,9.000000
200982,55561,2,54,3,33,7,9.818182
200983,55561,2,54,3,34,7,9.529412
200984,55561,2,54,3,35,7,9.257143
200985,55561,2,54,3,36,7,9.000000
200986,55561,2,58,3,37,7,9.405405
200987,55561,2,62,3,38,7,9.789474
200988,55561,2,63,3,39,7,9.692308


In [31]:
# Inspect dismissal types
display(
    df[df["is_wicket"] == True]
    .groupby("dismissal_type")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="count")
)

,dismissal_type,count
0,Caught,7327
1,Bowled,1689
2,LBW,666
3,RunOut,571
4,Stump,200
5,Hit Wicket,17
6,Retire Out,6
7,Obstracting the field,1
8,Obstructing the field,1


In [32]:
# Inspect all extras categories and their associated delivery styles
display(
    df.groupby(
        ["extras_type", "delivery_style"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(40)
)

,extras_type,delivery_style,count
175,NaN,Seamed In,20139
169,NaN,Orthodox,17485
165,NaN,Leg Break,14472
168,NaN,Off Cutter,13791
176,NaN,Seamed Out,13685
167,NaN,Off Break,13431
170,NaN,Out Swing,11503
162,NaN,Googly,11147
163,NaN,Inswing,11002
180,NaN,Straight,8848


In [33]:
# Identify wickets credited to the bowler


non_bowler_dismissals = [
    "runout",
    "retire out",
    "obstructing the field",
    "obstracting the field"
]

df["dismissal_clean"] = (
    df["dismissal_type"].str.strip().str.lower()
)

df["is_bowler_wicket"] = (
    df["is_wicket"].eq(True) &
    ~df["dismissal_clean"].isin(non_bowler_dismissals)
)

# Compare total wickets and bowler-creditable wickets
print("Total team wickets:", df["is_wicket"].sum())
print("Bowler-creditable wickets:", df["is_bowler_wicket"].sum())


Total team wickets: 10478
Bowler-creditable wickets: 9899


In [34]:
# Inspect only deliveries with extras

extras_check = (
    df[df["extras_type"].notna()]
    .groupby(
        ["extras_type", "delivery_style"],
        dropna=False
    )
    .agg(
        count=("match_id", "size"),
        total_runs=("total_runs", "sum"),
        wickets=("is_wicket", "sum")
    )
    .reset_index()
    .sort_values("count", ascending=False)
)

display(extras_check)


,extras_type,delivery_style,count,total_runs,wickets
123,wide,Bouncer,883,1113,1
130,wide,Inswing,617,874,0
135,wide,Off Cutter,614,670,3
85,legbyes,Seamed In,535,726,0
145,wide,Slow Bouncer,515,532,2
...,...,...,...,...,...
100,other,Flipper,1,1,0
31,Overstep,NaN,1,5,0
32,Penalty,Full Toss,1,5,0
56,byes,Slider,1,2,0


In [35]:
# Inspect unusual extras records in detail

special_extras = df[
    df["extras_type"].isin(["other", "Penalty", "Overstep"])
]

columns_to_check = [
    "match_id",
    "innings",
    "innings_ball_num",
    "striker_name",
    "bowler_name",
    "extras_type",
    "delivery_style",
    "runs",
    "extras1_runs",
    "extras_2_runs",
    "is_wicket",
    "dismissal_type"
]

display(
    special_extras[columns_to_check]
    .sort_values(["extras_type", "match_id"])
    .head(40)
)


,match_id,innings,innings_ball_num,striker_name,bowler_name,extras_type,delivery_style,runs,extras1_runs,extras_2_runs,is_wicket,dismissal_type
495,13513,1,20,Nicholas Pooran,George Garton,Overstep,Inswing,0,1,0,False,NaN
1229,13515,2,122,Sam Billings,Jason Holder,Overstep,Full Toss,1,1,0,False,NaN
1485,13523,1,4,Ishan Kishan,Akeal Hosein,Overstep,Orthodox,0,1,4,False,NaN
1486,13523,1,5,Ishan Kishan,Akeal Hosein,Overstep,Full Toss,1,1,0,False,NaN
3191,14094,1,1,Ruturaj Gaikwad,Umesh Yadav,Overstep,Inswing,0,1,0,False,NaN
4169,14098,1,5,Jos Buttler,Bhuvneshwar Kumar,Overstep,Out Swing,0,1,0,False,NaN
4181,14098,1,17,Yashasvi Jaiswal,Bhuvneshwar Kumar,Overstep,Yorker,0,1,0,False,NaN
4187,14098,1,23,Jos Buttler,Umran Malik,Overstep,Inswing,4,1,0,False,NaN
4192,14098,1,28,Yashasvi Jaiswal,Washington Sundar,Overstep,Off Break,1,1,0,False,NaN
4452,14099,1,33,Nitish Rana,Akash Deep,Overstep,Seamed Out,0,1,0,False,NaN


In [36]:
df.to_csv(
    "../outputs/ipl_t20i_regular_deliveries.csv",
    index=False
)